# 🧠 Fine-Tuned LLM for Patient FAQs
This notebook demonstrates how to use a fine-tuned language model from Hugging Face for answering frequently asked questions in healthcare.
It uses the `medalpaca` model (fine-tuned on medical QA) and a Gradio interface for interaction.

In [ ]:
!pip install gradio transformers --quiet

In [ ]:

import gradio as gr
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline


In [ ]:

import pandas as pd
import os
import tempfile

# History store
qa_history = []


In [ ]:

from PyPDF2 import PdfReader

def load_faqs_from_file(file):
    faqs = []
    filename = file.name
    ext = os.path.splitext(filename)[1].lower()
    if ext == '.csv':
        df = pd.read_csv(filename)
        for _, row in df.iterrows():
            faqs.append((str(row[0]), str(row[1]) if len(row) > 1 else ""))
    elif ext == '.pdf':
        reader = PdfReader(filename)
        text = "\n".join([page.extract_text() for page in reader.pages if page.extract_text()])
        for line in text.split("\n"):
            if '?' in line:
                q = line.strip()
                faqs.append((q, ""))
    return faqs


In [ ]:

def search_history(query):
    return "\n\n".join([entry for entry in qa_history if query.lower() in entry.lower()])

def export_history():
    with open("qa_history.txt", "w") as f:
        f.write("\n\n".join(qa_history))
    return "qa_history.txt"


In [ ]:

# Load a medically fine-tuned model
model_name = "medalpaca/medalpaca-7b"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, trust_remote_code=True)
qa_pipeline = pipeline("text-generation", model=model, tokenizer=tokenizer, max_new_tokens=256)


In [ ]:

with gr.Blocks() as demo:
    gr.Markdown("# 🩺 Patient FAQ Chatbot (Med-LLM)")

    with gr.Row():
        file_input = gr.File(label="Upload CSV or PDF with FAQs", type="file")
        upload_btn = gr.Button("Load FAQs")

    with gr.Row():
        question_input = gr.Textbox(lines=3, label="Ask a medical question (FAQ)")
        answer_output = gr.Textbox(label="Answer")

    with gr.Row():
        history_box = gr.Textbox(label="Previous Q&A", lines=10)

    with gr.Row():
        search_input = gr.Textbox(label="Search Q&A history")
        search_output = gr.Textbox(label="Search Results", lines=6)

    with gr.Row():
        export_btn = gr.Button("Export Q&A History")
        file_download = gr.File(label="Download Exported Q&A")

    def handle_qa(user_question):
        prompt = f"### Question: {user_question}\n### Answer:"
        response = qa_pipeline(prompt, do_sample=True, temperature=0.7)[0]["generated_text"]
        answer = response.split("### Answer:")[-1].strip()
        qa_history.append(f"Q: {user_question}\nA: {answer}")
        return answer, "\n\n".join(qa_history)

    def upload_faqs(file):
        faqs = load_faqs_from_file(file)
        if not faqs:
            return "No FAQs found in file."
        return f"{len(faqs)} FAQs loaded. Ask one or more now!"

    def handle_search(query):
        return search_history(query)

    def handle_export():
        path = export_history()
        return path

    upload_btn.click(upload_faqs, inputs=[file_input], outputs=[answer_output])
    question_input.submit(handle_qa, inputs=question_input, outputs=[answer_output, history_box])
    search_input.submit(handle_search, inputs=search_input, outputs=search_output)
    export_btn.click(handle_export, outputs=file_download)

demo.launch()
